This code takes the mean daily flow csv for all watersheds from 2000-2025 and finds the end date of the recession period when discharge hits Q5 and takes the recession start dates csv and sets all dates before the start date or after the end date to N/A.The final csv is the same daily Q file with each day as a row and each watershed as a column but with N/As for all dates outside the recession time period.

In [1]:
#imports
import pandas as pd
import datetime as dt
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import os
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter

In [ ]:
# inputs
working_directory=r"C:\Users\duffshan\Box\Shannon_Duffy\Writing\Code For Paper Publication"
# filepaths
dailyQ_file=os.path.join(working_directory,"Intermediate Outputs","Daily_meanQ_allsites_1950_2025.csv")
q5_file=os.path.join(working_directory,"Intermediate Outputs","5thPercentileFlows_nofall_2000_2025.csv")
start_date_file=os.path.join(working_directory,"Intermediate Outputs","Recession_start_date.csv")
outputfolder=os.path.join(working_directory,"Intermediate Outputs")

In [3]:
#set default font to arial
matplotlib.rc('font',family='Arial')

In [4]:
#read in daily Q
dailyQ=pd.read_csv(dailyQ_file)
dailyQ['DATE']=pd.to_datetime(dailyQ['DATE'])
dailyQ=dailyQ[dailyQ['WATERYEAR']>1999]
dailyQ=dailyQ[dailyQ['WATERYEAR']<2026]

#read in Q5
q5=pd.read_csv(q5_file)
q5=q5[q5['WATERYEAR']>1999]
q5=q5[q5['WATERYEAR']<2026]

#read in start dates
start_date_df=pd.read_csv(start_date_file)
start_date_df['GSLOOK']=pd.to_datetime(start_date_df['GSLOOK'])
start_date_df['GSWSMC']=pd.to_datetime(start_date_df['GSWSMC'])
start_date_df['GSWS01']=pd.to_datetime(start_date_df['GSWS01'])
start_date_df['GSWS02']=pd.to_datetime(start_date_df['GSWS02'])
start_date_df['GSWS03']=pd.to_datetime(start_date_df['GSWS03'])
start_date_df['GSWS08']=pd.to_datetime(start_date_df['GSWS08'])
start_date_df['GSWS09']=pd.to_datetime(start_date_df['GSWS09'])
start_date_df['GSWS10']=pd.to_datetime(start_date_df['GSWS10'])

C:\Users\duffshan\AppData\Local\Temp\ipykernel_3492\2480566845.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  start_date_df['GSLOOK']=pd.to_datetime(start_date_df['GSLOOK'])
C:\Users\duffshan\AppData\Local\Temp\ipykernel_3492\2480566845.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  start_date_df['GSWSMC']=pd.to_datetime(start_date_df['GSWSMC'])
C:\Users\duffshan\AppData\Local\Temp\ipykernel_3492\2480566845.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  start_date_df['GSWS01']=pd.to_datetime(start_date_df['GSWS01'])
C:\Users\duffshan\AppData\Local\Temp\ipy

In [5]:
all_q5date=pd.DataFrame()
for watershed in ["GSLOOK", "GSWS01","GSWS02","GSWS03","GSWS08","GSWS09","GSWS10","GSWSMC"]:
    ws_df=pd.DataFrame()
    for i in range(2000,2026):
        q5_value=q5.loc[q5['WATERYEAR'] == i, watershed]
        q5_value=q5_value.values[0]
        dailyQ_WY=dailyQ[dailyQ["WATERYEAR"]==i]
        dailyQ_WY=dailyQ_WY.reset_index()
        dailyQ_WY_trim=dailyQ_WY[dailyQ_WY["month"]<10]
        q5_index_list = dailyQ_WY_trim[dailyQ_WY_trim[watershed] <= q5_value]
        if len(q5_index_list)>0:
            q5_index=q5_index_list.index[0]
            q5_date=dailyQ_WY_trim.loc[q5_index, "DATE"]
        else:
            print(watershed,i,"recession doesn't hit Q5")
        if len(q5_index_list)>0:
            year_df=pd.DataFrame({"WATERYEAR":[i],f"{watershed}":[q5_date]})
        else:
            year_df=pd.DataFrame({"WATERYEAR":[i],f"{watershed}":np.nan})
        ws_df=pd.concat([ws_df,year_df])
    if watershed=="GSLOOK":
        all_q5date=pd.concat([all_q5date,ws_df])
    else:
        all_q5date=pd.merge(all_q5date,ws_df,how="outer",on="WATERYEAR")
print(all_q5date)
output_path = os.path.join(outputfolder, "Date_of_Q5.csv")
all_q5date.to_csv(output_path, index=False)

GSWS03 2020 recession doesn't hit Q5
    WATERYEAR     GSLOOK     GSWS01     GSWS02     GSWS03     GSWS08  \
0        2000 2000-08-24 2000-08-12 2000-09-13 2000-08-30 2000-08-29   
1        2001 2001-09-07 2001-08-12 2001-09-09 2001-09-08 2001-09-08   
2        2002 2002-09-09 2002-08-13 2002-09-07 2002-09-06 2002-09-08   
3        2003 2003-08-25 2003-08-19 2003-08-29 2003-08-25 2003-08-24   
4        2004 2004-08-16 2004-07-30 2004-08-18 2004-08-11 2004-08-14   
5        2005 2005-08-25 2005-08-15 2005-09-06 2005-09-03 2005-09-04   
6        2006 2006-09-02 2006-08-20 2006-09-04 2006-08-31 2006-08-29   
7        2007 2007-09-09 2007-08-03 2007-09-09 2007-09-11 2007-09-09   
8        2008 2008-09-11 2008-08-15 2008-09-14 2008-09-12 2008-09-12   
9        2009 2009-09-03 2009-08-19 2009-09-12 2009-09-12 2009-09-02   
10       2010 2010-08-19 2010-08-14 2010-08-25 2010-08-23 2010-08-22   
11       2011 2011-09-07 2011-09-04 2011-09-10 2011-09-08 2011-09-09   
12       2012 2012-09-14 20

C:\Users\duffshan\AppData\Local\Temp\ipykernel_3492\353839970.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ws_df=pd.concat([ws_df,year_df])


In [ ]:
#set discharge for all days to NaN if the day is before the recession start date or after the recession end date when discharge hits Q5
#make a copy of dailyQ
dailyQ_recession=dailyQ
#loop through each watershed and wateryear and find the start and end date of recession period
for watershed in ["GSLOOK", "GSWS01","GSWS02","GSWS03","GSWS08","GSWS09","GSWS10","GSWSMC"]:
    for i in range(2000,2026):
        q5_date=all_q5date.loc[all_q5date['WATERYEAR'] == i, watershed]
        q5_date=q5_date.values[0]
        start_date=start_date_df.loc[start_date_df['WATERYEAR'] == i, watershed]
        start_date=start_date.values[0]
        #set dates outside of period as NaN
        if not np.isnan(q5_date):
            dailyQ_recession.loc[(dailyQ['DATE'] < start_date) & (dailyQ['WATERYEAR'] ==i), watershed] = np.nan
            dailyQ_recession.loc[(dailyQ['DATE'] > q5_date) & (dailyQ['WATERYEAR'] ==i), watershed] = np.nan
print(dailyQ_recession)
#export to csv
output_path = os.path.join(outputfolder, "Filtered_Recession_Period_Q.csv")
dailyQ_recession.to_csv(output_path, index=False)

            DATE    GSLOOK        GSWS01    GSWS02    GSWS03        GSWS08  \
18262 1999-10-01  0.000005  3.826618e-07  0.000002  0.000002  7.106577e-07   
18263 1999-10-02  0.000005  3.935950e-07  0.000002  0.000002  6.013258e-07   
18264 1999-10-03  0.000005  3.607955e-07  0.000002  0.000002  6.013258e-07   
18265 1999-10-04  0.000005  3.279959e-07  0.000002  0.000002  5.466598e-07   
18266 1999-10-05  0.000005  4.591942e-07  0.000002  0.000002  5.903926e-07   
...          ...       ...           ...       ...       ...           ...   
27754 2025-09-26  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27755 2025-09-27  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27756 2025-09-28  0.000005  7.433160e-07  0.000003  0.000002  1.455536e-07   
27757 2025-09-29  0.000005  7.620587e-07  0.000003  0.000002  1.830907e-07   
27758 2025-09-30  0.000005  9.711778e-07  0.000003  0.000002  3.170202e-07   

             GSWS09        GSWS10    GSWSMC        date  year  